# 00 — Download Protocols & Extract SoA Pages

Downloads protocol PDFs from ClinicalTrials.gov, extracts SoA pages as
separate PDFs, and converts full protocols to Markdown via pymupdf4llm.

**Input:** `studies_protocols.xlsx` — `protocols` sheet with columns:
`nct_id`, `soa_pages`, `protocol_url`.

**Progressive execution** — skips any file that already exists.
Re-run safely to fill in only what's missing.

**Output per protocol:**
- `{NCT_ID}.pdf` — full protocol from ClinicalTrials.gov
- `{NCT_ID}_soa.pdf` — SoA pages only (requires `soa_pages` in Excel)
- `{NCT_ID}.md` — full protocol as markdown
- `SoA2USDM/extracted/` — folder scaffold ready for extraction workflow

## Configuration

In [ ]:
import sys
from pathlib import Path

# Make the repo-root package importable (or `pip install -e .` from the repo root)
sys.path.insert(0, str(Path.cwd().parent))

from soa2usdm import config

# === CONFIGURE ===
COLLECTION = 'usdm_data'

COLLECTION_ROOT = config.get_collection_root(COLLECTION)
EXCEL_FILE = COLLECTION_ROOT / 'studies_protocols.xlsx'
EXCEL_SHEET = 'protocols'
OUTPUT_ROOT = config.get_collection_path(COLLECTION)  # .../protocols/
DELAY_SECONDS = 2

# ── Verify ────────────────────────────────────────────────────
assert COLLECTION_ROOT.exists(), f'Collection not found: {COLLECTION_ROOT}'
assert EXCEL_FILE.exists(), f'Excel not found: {EXCEL_FILE}'
print(f'Repo root:   {config.REPO_ROOT}')
print(f'Collection:  {COLLECTION}')
print(f'Excel:       {EXCEL_FILE}')
print(f'Output:      {OUTPUT_ROOT}')

In [ ]:
# Uncomment if not installed:
# !pip install pymupdf4llm

## Imports

In [ ]:
import pandas as pd
import requests
import pymupdf
import pymupdf4llm
from time import sleep

SESSION = requests.Session()
SESSION.headers.update({
    'User-Agent': 'Mozilla/5.0 (research protocol download)'
})

## Read protocol metadata

In [ ]:
df = pd.read_excel(EXCEL_FILE, sheet_name=EXCEL_SHEET)
print(f'Protocols to process: {len(df)}')

# ── Flag protocols with no SoA pages ──────────────────────────
no_soa = df['soa_pages'].isna() | (df['soa_pages'].astype(str).str.strip() == '')
if no_soa.any():
    print(f'')
    print(f'⚠ WARNING: {no_soa.sum()} protocol(s) have no SoA pages defined:')
    for _, row in df[no_soa].iterrows():
        print(f'    {row["nct_id"]} — SoA extract will be SKIPPED')
    print(f'  Add page ranges to the \'{EXCEL_SHEET}\' sheet and re-run.')

df

## Processing functions

Each function checks whether its output file already exists and returns
early if so — making the entire notebook progressive.

In [ ]:
def download_pdf(url: str, dest: Path) -> bool:
    """Download PDF from CT.gov. Skips if dest exists."""
    if dest.exists():
        print(f'  ✓ exists: {dest.name}')
        return True
    print(f'  ↓ downloading: {url}')
    resp = SESSION.get(url, timeout=60)
    resp.raise_for_status()
    dest.write_bytes(resp.content)
    print(f'  ✓ saved: {dest.name} ({len(resp.content):,} bytes)')
    return True


def extract_soa(src: Path, dest: Path, soa_str: str) -> bool:
    """Extract SoA page range into separate PDF. Skips if dest exists."""
    if dest.exists():
        print(f'  ✓ exists: {dest.name}')
        return True
    if pd.isna(soa_str) or str(soa_str).strip() == '':
        print(f'  ✗ ERROR: no SoA pages defined — cannot extract')
        return False
    parts = str(soa_str).strip().split('-')
    start = int(parts[0]) - 1
    end = int(parts[1]) - 1
    doc = pymupdf.open(src)
    soa_doc = pymupdf.open()
    soa_doc.insert_pdf(doc, from_page=start, to_page=end)
    soa_doc.save(dest)
    soa_doc.close()
    doc.close()
    print(f'  ✓ SoA: pages {start+1}–{end+1} → {dest.name}')
    return True


def convert_to_markdown(src: Path, dest: Path) -> bool:
    """Convert PDF to markdown. Skips if dest exists."""
    if dest.exists():
        print(f'  ✓ exists: {dest.name}')
        return True
    md_text = pymupdf4llm.to_markdown(str(src))
    dest.write_text(md_text, encoding='utf-8')
    print(f'  ✓ markdown: {dest.name} ({len(md_text):,} chars)')
    return True

## Process all protocols

In [ ]:
results = []

for _, row in df.iterrows():
    nct = row['nct_id']
    url = row['protocol_url']
    soa_str = row['soa_pages']

    print(f'\n{"=" * 60}')
    print(f'{nct}')
    print(f'{"=" * 60}')

    folder = OUTPUT_ROOT / nct
    folder.mkdir(parents=True, exist_ok=True)
    (folder / 'SoA2USDM' / 'extracted').mkdir(parents=True, exist_ok=True)

    full_pdf = folder / f'{nct}.pdf'
    soa_pdf = folder / f'{nct}_soa.pdf'
    md_file = folder / f'{nct}.md'

    status = {'nct_id': nct, 'downloaded': False, 'soa': False, 'markdown': False}

    # 1. Download
    try:
        status['downloaded'] = download_pdf(url, full_pdf)
    except Exception as e:
        print(f'  ✗ download failed: {e}')
        results.append(status)
        continue

    # 2. SoA extract
    try:
        status['soa'] = extract_soa(full_pdf, soa_pdf, soa_str)
    except Exception as e:
        print(f'  ✗ SoA extraction failed: {e}')

    # 3. Markdown
    try:
        status['markdown'] = convert_to_markdown(full_pdf, md_file)
    except Exception as e:
        print(f'  ✗ markdown conversion failed: {e}')

    results.append(status)
    sleep(DELAY_SECONDS)

print(f'\n{"=" * 60}')
print('Done.')

## Summary

In [ ]:
summary = pd.DataFrame(results)
print(f'Downloaded:  {summary["downloaded"].sum()}/{len(summary)}')
print(f'SoA extract: {summary["soa"].sum()}/{len(summary)}')
print(f'Markdown:    {summary["markdown"].sum()}/{len(summary)}')

# ── Highlight failures ────────────────────────────────────────
failed_soa = summary[~summary['soa']]
if len(failed_soa) > 0:
    print(f'')
    print(f'⚠ {len(failed_soa)} protocol(s) missing SoA extract:')
    for _, r in failed_soa.iterrows():
        print(f'    {r["nct_id"]}')

summary

## Verify output

In [ ]:
for folder in sorted(OUTPUT_ROOT.iterdir()):
    if folder.is_dir():
        print(f'\n{folder.name}/')
        for f in sorted(folder.iterdir()):
            size_kb = f.stat().st_size / 1024
            print(f'  {f.name:40s}  {size_kb:8.1f} KB')